# Sigue el diseño y las particiones de validación sin fuga

Nivel: 2. Requisitos previos: Nivel 1: Construye un ejemplo de aprendizaje para procesamiento de minerales. Tiempo de lectura: 32 minutos.

## Objetivos

Genera un diseño determinista pequeño con el pipeline OreFlow, inspecciona particiones de interpolación y caso completo no visto, y relaciónalas con la validación comprometida de tamaño completo.

## Preparación

```bash
pip install numpy==2.2.6 scipy==1.17.1 pandas==2.2.3 matplotlib==3.11.2 nbclient==0.11.0 nbformat==5.11.1 ipykernel==7.4.0 SALib==1.6.0
```

El pipeline OreFlow se importa desde `data-pipeline/`; este notebook nunca instala paquetes.

Configura las importaciones y registra versiones. Este paso hace reproducible la ejecución del método.

In [1]:
# Importa herramientas de análisis y registra sus versiones exactas.
import importlib.metadata
import json
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Resuelve el pipeline local desde la estructura relativa del repositorio.
repo = Path('../../../').resolve()
sys.path.insert(0, str(repo / 'data-pipeline'))
for package in ('numpy', 'scipy', 'pandas', 'matplotlib', 'nbclient', 'nbformat', 'ipykernel'):
    print(package, importlib.metadata.version(package))

numpy 2.2.6
scipy 1.17.1
pandas 2.2.3
matplotlib 3.11.2
nbclient 0.11.0
nbformat 5.11.1
ipykernel 7.4.0


## Datos

Responsable: autores de OreFlow; `data/derived/contract/operating_contract.json` y `data/derived/learning.json`, licencia MIT del proyecto. Las filas son simulaciones de escenarios propios, no mediciones de planta. Repositorio: https://github.com/fsantibanezleal/CAOS_OreFlow.

## Método

El constructor muestrea entradas factibles dentro del Contrato 1, ejecuta el motor y crea objetivos y características físicas. Interpolación retiene filas con semilla dentro de cada caso. Leave-one-case-out retiene un caso completo; la transferencia por grupo retiene también casos hermanos. No son protocolos intercambiables: el segundo mide transferencia a un caso nunca visto.

Construye un diseño pequeño con semilla para dos casos. OreFlow genera cada fila con su constructor y solver de producción; no se ingresan datos manualmente.

In [2]:
from pipeline.cases.catalog import CASES
from pipeline.methods.learning import build_design, interpolation_split, leave_one_case_out

# Paso ejecutable del método OreFlow explicado en la celda anterior.
contract = json.loads((repo / 'data/derived/contract/operating_contract.json').read_text(encoding='utf-8'))
selected = tuple(item for item in CASES if item.id in ('copper_porphyry_soft', 'iron_magnetite_fine'))
# Paso ejecutable del método OreFlow explicado en la celda anterior.
import contextlib
import io
with contextlib.redirect_stdout(io.StringIO()):
    design = build_design(contract, cases=selected, per_case=8, seed=20261026)
x, y, case_of = design['x'], design['y'], design['case']
print('matriz de diseño:', x.shape, '| target matrix:', y.shape)
print('filas por caso:', {str(name): int((case_of == name).sum()) for name in dict.fromkeys(case_of)})

matriz de diseño: (16, 22) | target matrix: (16, 3)
filas por caso: {'copper_porphyry_soft': 8, 'iron_magnetite_fine': 8}


Crea ambos protocolos de validación y verifica sus límites contra fuga. Los casos de prueba de un fold de caso completo no pueden aparecer en los índices de entrenamiento.

In [3]:
# Crea ambos protocolos de validación y verifica sus límites contra fuga.
train_idx, test_idx = interpolation_split(case_of, 0.25, seed=20261026)
folds = leave_one_case_out(case_of)
print('interpolation train/test:', len(train_idx), len(test_idx))
print('whole-folds por caso:', [(name, len(train), len(test)) for name, train, test in folds])
assert set(case_of[train_idx]) == set(case_of[test_idx])
assert all(set(case_of[train]).isdisjoint(set(case_of[test])) for _, train, test in folds)

interpolation train/test: 12 4
whole-folds por caso: [('copper_porphyry_soft', 8, 8), ('iron_magnetite_fine', 8, 8)]


Lee la evaluación de producción por separado de esta muestra didáctica. El registro comprometido se horneó con los doce casos y la configuración completa declarada.

In [4]:
# Lee la evaluación de producción por separado de esta muestra didáctica.
record = json.loads((repo / 'data/derived/learning.json').read_text(encoding='utf-8'))
print('versión del motor:', record['engine_version'])
print('production design rows:', record['design']['rows'])
print('interpolation train/test:', record['interpolation']['train_rows'], record['interpolation']['test_rows'])
print('case holdouts:', len(record['leave_one_case_out']))
print('ore-group holdouts:', len(record['leave_one_group_out']))

versión del motor: 0.10.004
production design rows: 3072
interpolation train/test: 2460 612
case holdouts: 12
ore-group holdouts: 2


## Resultados

El diseño pequeño hace inspeccionables las particiones; el registro completo aporta métricas de ajuste. Interpolación estima comportamiento dentro de casos representados; retener casos o grupos prueba transferencia. Ninguno es validación de planta si las etiquetas proceden del motor.

## Resumen

Ejecutaste las funciones reales de diseño y partición y verificaste los límites de fuga. Mantén separada la muestra ilustrativa de la evaluación de bake completo.

## Referencias

- https://doi.org/10.1023/A:1010933404324
- https://doi.org/10.1214/aos/1013203451
- https://doi.org/10.1023/A:1010933404324